In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQFAM_G.xpt"
csv_file = DATA_DIR / "SMQFAM_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 SMQFAM_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMD410",
    "SMD415",
    "SMD415A",
    "SMD430"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQFAM_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9756:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 5:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,756 rows x 5 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached SMQFAM_G.xpt:
#
#     tiny artifacts = 0
#
# Therefore NO tiny -> zero restoration is required.
#
# IMPORTANT:
#
# If another/local copy contains tiny values,
# STOP rather than automatically replacing them.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in SMQFAM_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total > 0:

    print(
        "\nOrdinary zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected numeric zeros found "
        "in attached SMQFAM_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No ordinary numeric zeros."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Always check this BEFORE Int64 conversion.
#
# Attached SMQFAM_G contains:
#
#     genuine fractional variables = NONE
#
# All five fields are identifiers, codes, or
# whole-number counts.
#
# If a future/local copy contains a genuine fraction:
#
#     STOP
#
# Do NOT round it.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in SMQFAM_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# SMQFAM_G contains NO genuine decimal variables.
#
# Therefore all five fields can safely become
# pandas nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used for conversion.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely become Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 5 SMQFAM_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 8. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 9. CDC CHECK - SMD410
# ============================================================

#
# Does anyone who lives here smoke cigarettes,
# cigars or pipes anywhere inside the home?
#
# CDC:
#
# 1 = Yes       1,229
# 2 = No        8,486
# 7 = Refused       0
# 9 = Don't know    0
# Missing           41
#

check_frequency(
    "SMD410",
    {
        1: 1229,
        2: 8486
    },
    41
)


# ============================================================
# 10. CDC CHECK - SMD415
# ============================================================

#
# Total number of smokers inside home.
#
# Includes smokers of all tobacco types.
#
# CDC:
#
# 1 = 1 smoker       738
# 2 = 2 smokers      386
# 3 = 3 or more      105
# Missing          8,527
#

check_frequency(
    "SMD415",
    {
        1: 738,
        2: 386,
        3: 105
    },
    8527
)


# ============================================================
# 11. SMD410 -> SMD415 ROUTING CHECK
# ============================================================

#
# SMD415 should be populated exactly when:
#
#     SMD410 == 1
#
# Because these are nullable Int64 fields,
# explicitly turn pd.NA into False for the
# temporary Boolean validation mask.
#
# THIS DOES NOT MODIFY THE DATA.
#

smd410_yes = (
    df["SMD410"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


smd415_nonmissing = (
    df["SMD415"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


routing_mismatch = int(
    np.sum(
        smd410_yes
        !=
        smd415_nonmissing
    )
)


print("\n--- SMD410 -> SMD415 ROUTING ---")


print(
    "Routing mismatches:",
    routing_mismatch
)


if routing_mismatch != 0:

    raise ValueError(
        "SMD415 nonmissing positions do not "
        "exactly match SMD410=1."
    )


print(
    "PASS: SMD415 routing exactly matches SMD410=1."
)


# ============================================================
# 12. CDC CHECK - SMD415A
# ============================================================

#
# Total number of cigarette smokers inside home.
#
# CDC:
#
# 1 = 1 cigarette smoker      710
# 2 = 2 cigarette smokers     368
# 3 = 3 or more                92
# Missing                   8,586
#

check_frequency(
    "SMD415A",
    {
        1: 710,
        2: 368,
        3: 92
    },
    8586
)


# ============================================================
# 13. SMD415A CONSISTENCY CHECK
# ============================================================

#
# SMD410=Yes:
#
#     1,229 households
#
# SMD415A populated:
#
#     1,170 households
#
# Therefore:
#
#     59 households reported someone smoking
#     inside the home, but did not have a
#     cigarette-smoker count.
#
# This can occur because SMD410 includes
# cigarettes, cigars, OR pipes.
#

smoking_household_count = int(
    df["SMD410"]
    .eq(1)
    .sum()
)


cigarette_smoker_count_recorded = int(
    df["SMD415A"]
    .notna()
    .sum()
)


smoking_household_without_cigarette_count = int(
    (
        df["SMD410"]
        .eq(1)
        .fillna(False)
        &
        df["SMD415A"]
        .isna()
    )
    .sum()
)


print("\n--- SMD415A CONSISTENCY ---")


print(
    "SMD410=Yes:",
    smoking_household_count
)


print(
    "SMD415A nonmissing:",
    cigarette_smoker_count_recorded
)


print(
    "SMD410=Yes but SMD415A missing:",
    smoking_household_without_cigarette_count
)


if smoking_household_count != 1229:

    raise ValueError(
        "Unexpected SMD410 Yes count."
    )


if cigarette_smoker_count_recorded != 1170:

    raise ValueError(
        "Unexpected SMD415A nonmissing count."
    )


if smoking_household_without_cigarette_count != 59:

    raise ValueError(
        "Unexpected SMD410/SMD415A routing pattern."
    )


print(
    "PASS: SMD415A household-smoking pattern validated."
)


# ============================================================
# 14. CDC CHECK - SMD430
# ============================================================

#
# Total number of cigarettes smoked per day
# anywhere inside the home.
#
# CDC documentation:
#
#     1-35 = 1,058
#     40   =   112
#     Missing = 8,586
#
# CDC instructions say:
#
#     1 pack = 20 cigarettes
#     if none, enter 0
#     if less than 1/day, enter 1
#
# The released attached file contains NO zero values.
#
# DO NOT create zeros.
#

s = df["SMD430"]


valid_1_to_35 = int(
    s.between(
        1,
        35
    )
    .sum()
)


topcoded_40 = int(
    s.eq(40)
    .sum()
)


missing_smd430 = int(
    s.isna()
    .sum()
)


zero_smd430 = int(
    s.eq(0)
    .sum()
)


unexpected_36_to_39 = int(
    s.between(
        36,
        39
    )
    .sum()
)


print("\n--- SMD430 ---")


print(
    "1-35 cigarettes/day:",
    valid_1_to_35
)


print(
    "40 or more:",
    topcoded_40
)


print(
    "Zeros:",
    zero_smd430
)


print(
    "Missing:",
    missing_smd430
)


if not (
    valid_1_to_35 == 1058
    and
    topcoded_40 == 112
    and
    zero_smd430 == 0
    and
    unexpected_36_to_39 == 0
    and
    missing_smd430 == 8586
):

    raise ValueError(
        "SMD430 differs from CDC/released XPT pattern."
    )


print(
    "PASS: SMD430 matches CDC release."
)


# ============================================================
# 15. SMD415A -> SMD430 ROUTING CHECK
# ============================================================

#
# In the attached release, SMD430 has exactly the
# same nonmissing positions as SMD415A.
#

smd415a_nonmissing = (
    df["SMD415A"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


smd430_nonmissing = (
    df["SMD430"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


routing_mismatch = int(
    np.sum(
        smd415a_nonmissing
        !=
        smd430_nonmissing
    )
)


print("\n--- SMD415A -> SMD430 ROUTING ---")


print(
    "Routing mismatches:",
    routing_mismatch
)


if routing_mismatch != 0:

    raise ValueError(
        "SMD415A and SMD430 missing patterns differ."
    )


print(
    "PASS: SMD430 nonmissing positions exactly "
    "match SMD415A."
)


# ============================================================
# 16. ALLOWED RELEASED VALUE CHECK
# ============================================================

#
# Additional protection against unexpected
# questionnaire values.
#

ALLOWED_VALUES = {

    "SMD410": {
        1,
        2
    },

    "SMD415": {
        1,
        2,
        3
    },

    "SMD415A": {
        1,
        2,
        3
    },

    "SMD430": (
        set(
            range(
                1,
                36
            )
        )
        |
        {
            40
        }
    )
}


print("\n--- ALLOWED VALUE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected values: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only expected values."
    )


# ============================================================
# 17. FINAL ZERO-PATTERN CHECK
# ============================================================

#
# This attached SMQFAM_G contains NO legitimate
# released zeros.
#
# Therefore the corrected dataframe should still
# contain zero total numeric zeros.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")


print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zeros exist in final dataframe:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero restoration was required."
)


# ============================================================
# 18. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# SMQFAM_G requires ZERO numeric corrections.
#
# Therefore every numeric value must remain exactly
# equal to the source XPT.
#
# The ONLY intended change is textual formatting:
#
#     62161.0 -> 62161
#     1.0     -> 1
#     40.0    -> 40
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 5 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# IMPORTANT STANDARD RULE:
#
# NO rounding.
#
# NO:
#
#     float_format=
#
# SMQFAM_G happens to contain no decimal variables,
# but keeping the same export pattern protects your
# overall NHANES workflow.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")


print(
    csv_file
)


# ============================================================
# 20. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 21. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 5 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 22. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 23. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 24. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 25. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        numeric_values
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO CHECK ---")


print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zeros found in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no unexpected zeros."
)


# ============================================================
# 26. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# SMQFAM_G contains NO genuine decimal variables.
#
# Therefore every populated field in THIS FILE
# should serialize without unnecessary ".0".
#
# Examples:
#
#     62161.0 -> 62161
#     1.0     -> 1
#     40.0    -> 40
#
# This global .0 check is safe for SMQFAM_G only
# because all five fields are proven whole-number.
#
# DO NOT apply this globally to Dietary, Lab,
# Examination, sample-weight, or other genuine
# decimal variables.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting "
    "in any SMQFAM_G field."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 29. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "SMQFAM_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN minimum:",
    df["SEQN"].min()
)


print(
    "SEQN maximum:",
    df["SEQN"].max()
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Original ordinary zeros:",
    true_zero_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "SMD410 Yes households:",
    int(
        df["SMD410"]
        .eq(1)
        .sum()
    )
)


print(
    "SMD415 nonmissing:",
    int(
        df["SMD415"]
        .notna()
        .sum()
    )
)


print(
    "SMD415A nonmissing:",
    int(
        df["SMD415A"]
        .notna()
        .sum()
    )
)


print(
    "SMD430 nonmissing:",
    int(
        df["SMD430"]
        .notna()
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 SMQFAM_G XPT -> CSV
Rows: 9,756
Columns: 5
PASS: Structure = 9,756 rows x 5 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 0
PASS: No tiny XPORT artifacts.
PASS: No ordinary numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
SMD410: 0 fractional observations
SMD415: 0 fractional observations
SMD415A: 0 fractional observations
SMD430: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

PASS: All 5 SMQFAM_G variables converted to nullable Int64 without rounding.
PASS: SMD410 matches CDC.
PASS: SMD415 matches CDC.

--- SMD410 -> SMD415 ROUTING ---
Routing mismatches: 0
PASS: SMD415 routing exactly matches SMD410=1.
PASS: SMD415A matches CDC.

--- SMD415A CONSISTENCY ---
SMD410=Yes: 1229
SMD415A nonmissing: 1170
SMD410=Yes but SMD415A missing: 59
PASS: SMD415A household-smoking pattern validated.

--